# Prepare PeMSD4 road traffic

Prepare the five-minute, 15-minute, hourly, and daily flow series for sensor
indices `62`, `103`, and `240`. Read channel `0` from `pemsd4.npz`, treat zeros
as missing, and interpolate each sensor's series before sum aggregation.

See [Dataset preparation](../docs/DATASETS.md) for the source and experiment
settings. Run the cells in order from the repository root or `notebooks/`
directory.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent)
     if (path / "tools" / "prepare_datasets.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tools.prepare_datasets import write_outputs

import numpy as np

## Input and output

Place `pemsd4.npz` in `data/Road/`, or update `INPUT_PATH` below.
Set `OVERWRITE = True` to replace existing prepared CSVs.

In [ ]:
INPUT_PATH = ROOT / "data" / "Road" / "pemsd4.npz"
OUTPUT_DIR = ROOT / "data" / "Road"
SENSORS = (62, 103, 240)
OVERWRITE = False

## Prepare the flow series

The preparation uses linear interpolation in both directions across the
series, including endpoint filling. This step precedes the training split.

In [ ]:
with np.load(INPUT_PATH) as archive:
    raw = archive["data"]
if raw.ndim != 3 or raw.shape[1] <= max(SENSORS) or raw.shape[2] == 0:
    raise ValueError(f"Expected a [time, sensors, channels] array, got {raw.shape}.")

flow = raw[:, :, 0].astype(float)
flow[flow == 0] = np.nan
flow = pd.DataFrame(flow).interpolate(
    method="linear", axis=0, limit_direction="both"
)

timestamps = pd.date_range(
    start="2018-01-01 00:00:00",
    periods=len(flow),
    freq="5min",
    name="date",
)

## Aggregate and save

Use the same sensor indices at every resolution. Save each series with
`date` and `value` columns and the filenames expected by the launchers.

In [ ]:
outputs = {}
for sensor in SENSORS:
    finest = pd.DataFrame({"value": flow[sensor].to_numpy()}, index=timestamps)
    for resolution, rule in (
        ("5m", None), ("15m", "15min"), ("1h", "1h"), ("1d", "1D")
    ):
        frame = finest if rule is None else finest.resample(
            rule, origin="start_day", label="left", closed="left"
        ).sum()
        outputs[f"road_sensor_{sensor}_{resolution}.csv"] = frame

In [ ]:
write_outputs(outputs, OUTPUT_DIR, overwrite=OVERWRITE)

In [ ]:
summary = pd.DataFrame([
    {"file": name, "rows": len(frame),
     "start": frame.index[0], "end": frame.index[-1]}
    for name, frame in outputs.items()
])
summary

## Training split

The study's road-traffic test period starts at `2018-02-17 00:00:00`.
The loader uses the final 10% of the preceding observations for validation
and fits normalization on the remaining training observations.